# CCI polynomial Ridge regression

This experiment forecasts Climate Comfort Index (CCI) at D+1, D+2, and D+3 from the same leakage-safe 86-feature table used by the tree models.

A degree-2 polynomial expansion creates 3,827 terms. Ridge regularisation is selected by **validation MAE only**; the chronological held-out test split is not loaded.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler


def project_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Run this notebook from the experiments repository.")


PROJECT_ROOT = project_root()
DATA_DIR = PROJECT_ROOT / "data" / "processed" / "comfort_climate"
MODEL_DIR = PROJECT_ROOT / "models" / "comfort_climate" / "experiments"
FIGURE_DIR = PROJECT_ROOT / "reports" / "figures" / "comfort_climate"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
EXPERIMENT = "cci_polynomial_regression"
HORIZONS = (1, 2, 3)
ALPHAS = (0.1, 1.0, 10.0, 100.0, 1_000.0, 10_000.0)

In [ ]:
metadata = json.loads((DATA_DIR / "cci_feature_metadata.json").read_text())
FEATURE_COLUMNS = metadata["feature_columns"]
TARGET_COLUMNS = metadata["target_columns"]
FEATURE_GROUPS = metadata["feature_groups"]

train = pd.read_csv(DATA_DIR / "sydney_cci_train.csv", parse_dates=["feature_date"]).set_index("feature_date")
validation = pd.read_csv(DATA_DIR / "sydney_cci_val.csv", parse_dates=["feature_date"]).set_index("feature_date")
assert train.index.max() < validation.index.min()
assert set(FEATURE_COLUMNS).issubset(train.columns)

X_train = train[FEATURE_COLUMNS]
X_validation = validation[FEATURE_COLUMNS]
print(f"Training rows: {len(train):,}; validation rows: {len(validation):,}")
print(f"Original features: {len(FEATURE_COLUMNS)}; degree-2 terms: 3,827")

In [ ]:
def regression_metrics(actual: pd.Series, predicted: np.ndarray) -> dict[str, float]:
    return {
        "mae": float(mean_absolute_error(actual, predicted)),
        "rmse": float(mean_squared_error(actual, predicted) ** 0.5),
        "r2": float(r2_score(actual, predicted)),
        "pred_std": float(np.std(predicted)),
    }


# Baselines use information available on the feature date only.
def climatology_profile(frame: pd.DataFrame, target: str, window: int = 15) -> pd.Series:
    """Return a year-wrapped, smoothed day-of-year CCI profile from training data only."""
    profile = frame.groupby(frame.index.dayofyear)[target].mean().reindex(range(1, 367))
    half_window = window // 2
    wrapped = pd.concat([profile.iloc[-half_window:], profile, profile.iloc[:half_window]])
    return wrapped.rolling(window, center=True, min_periods=1).mean().iloc[half_window:-half_window]


CLIMATOLOGY = {target: climatology_profile(train, target) for target in TARGET_COLUMNS}
validation_climatology = {
    target: validation.index.dayofyear.map(CLIMATOLOGY[target]).to_numpy()
    for target in TARGET_COLUMNS
}

base_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("polynomial", PolynomialFeatures(degree=2, include_bias=False)),
    ("scale", StandardScaler()),
    ("ridge", Ridge()),
])

models, predictions, selected_alphas, rows = {}, {}, {}, []
for horizon in HORIZONS:
    target = f"cci_t{horizon}"
    y_train, y_validation = train[target], validation[target]
    candidates = {}
    for alpha in ALPHAS:
        candidate = clone(base_pipeline).set_params(ridge__alpha=alpha)
        candidate.fit(X_train, y_train)
        candidates[alpha] = candidate
    best_alpha = min(candidates, key=lambda alpha: mean_absolute_error(y_validation, candidates[alpha].predict(X_validation)))
    model = candidates[best_alpha]
    prediction = model.predict(X_validation)
    models[horizon] = model
    predictions[horizon] = prediction
    selected_alphas[horizon] = best_alpha
    rows.extend([
        {"horizon": horizon, "model": "Polynomial Ridge", **regression_metrics(y_validation, prediction)},
        {"horizon": horizon, "model": "Persistence", **regression_metrics(y_validation, validation["cci"].to_numpy())},
        {"horizon": horizon, "model": "Climatology", **regression_metrics(y_validation, validation_climatology[target])},
    ])

results = pd.DataFrame(rows).set_index(["horizon", "model"])
print("Selected Ridge alpha per horizon:", selected_alphas)
results

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 7), constrained_layout=True)
for column, horizon in enumerate(HORIZONS):
    observed = validation[f"cci_t{horizon}"]
    predicted = predictions[horizon]
    axes[0, column].scatter(observed, predicted, alpha=0.45, s=12)
    axes[0, column].plot([0, 100], [0, 100], "k--")
    axes[0, column].set(title=f"D+{horizon}: observed vs predicted", xlabel="Observed CCI", ylabel="Predicted CCI")
    axes[1, column].scatter(predicted, observed - predicted, alpha=0.45, s=12)
    axes[1, column].axhline(0, color="k", linestyle="--")
    axes[1, column].set(title=f"D+{horizon}: residuals", xlabel="Predicted CCI", ylabel="Observed − predicted")
fig.savefig(FIGURE_DIR / f"{EXPERIMENT}-diagnostics.png", dpi=150)
plt.show()

In [ ]:
sys.path.insert(0, str(PROJECT_ROOT / "36120_26sp_at2_25945320_experiments"))
from modeling.deployment import export_tabular_bundle

experiment_metadata = {
    "experiment": EXPERIMENT,
    "student_id": "25945320",
    "algorithm": "Degree-2 Polynomial Ridge Regression",
    "prediction_type": "regression",
    "feature_columns": FEATURE_COLUMNS,
    "feature_column_order_is_significant": True,
    "polynomial_degree": 2,
    "expanded_feature_count": 3827,
    "selected_alphas": selected_alphas,
    "validation_metrics": {
        f"cci_t{h}": results.loc[(h, "Polynomial Ridge")].drop("pred_std").to_dict()
        for h in HORIZONS
    },
    "baseline_metrics": {
        f"cci_t{h}": {
            baseline: results.loc[(h, baseline)].to_dict()
            for baseline in ("Persistence", "Climatology")
        }
        for h in HORIZONS
    },
    "split_boundaries": metadata["split_boundaries"],
    "embargo_days": metadata["embargo_days"],
    "evaluated_on": ["train", "val"],
    "held_out": ["test"],
}
bundle_dir = export_tabular_bundle(
    output_dir=MODEL_DIR / EXPERIMENT,
    experiment=EXPERIMENT,
    models=models,
    feature_columns=FEATURE_COLUMNS,
    metadata=experiment_metadata,
    model_family="polynomial_ridge",
)
print(f"Saved API-compatible bundle to {bundle_dir.relative_to(PROJECT_ROOT)}")
print("Review validation results before running promote_production.py.")

## 4. Feature-group ablation

Each cumulative feature family is scored on chronological validation data. This shows whether each engineered family improves prediction beyond same-day CCI persistence.

In [ ]:
FEATURE_GROUPS = metadata["feature_groups"]
GROUP_ORDER = [
    "same_day_observations", "same_day_scores", "lags", "rolling_statistics",
    "rolling_precipitation", "rolling_gusts", "deltas", "seasonal",
]
ordered_groups = [group for group in GROUP_ORDER if group in FEATURE_GROUPS]
ordered_groups += [group for group in FEATURE_GROUPS if group not in ordered_groups]

ablation_configs = {"persistence only": ["cci"]}
cumulative = []
for group in ordered_groups:
    cumulative = list(dict.fromkeys([*cumulative, *FEATURE_GROUPS[group]]))
    ablation_configs[group] = cumulative

ablation_rows = []
for label, columns in ablation_configs.items():
    for horizon in HORIZONS:
        target = f"cci_t{horizon}"
        candidate = clone(models[horizon]).fit(train[columns], train[target])
        ablation_rows.append({
            "config": label,
            "target": target,
            "mae": mean_absolute_error(validation[target], candidate.predict(validation[columns])),
        })

ablation = pd.DataFrame(ablation_rows)
ablation_mae = ablation.pivot(index="config", columns="target", values="mae").reindex(ablation_configs)
ablation_mae.insert(0, "n_features", [len(columns) for columns in ablation_configs.values()])
print("Validation MAE by cumulative feature set (lower is better):\n")
print(ablation_mae.to_string())

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5), constrained_layout=True)
ablation_mae[[f"cci_t{h}" for h in HORIZONS]].plot(marker="o", ax=axes[0])
for horizon, style in zip(HORIZONS, ("--", "-.", ":")):
    axes[0].axhline(
        results.loc[(horizon, "Persistence"), "mae"], color=f"C{horizon - 1}", linestyle=style, alpha=0.55
    )
axes[0].set(title="Validation MAE as families are added", ylabel="MAE (CCI)", xlabel="")
(-ablation_mae[[f"cci_t{h}" for h in HORIZONS]].diff()).plot.bar(ax=axes[1])
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set(title="MAE reduction contributed by each family", ylabel="MAE reduction", xlabel="")
fig.savefig(FIGURE_DIR / f"{EXPERIMENT}-ablation.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Validation predictions and feature attribution

Attribution is based on absolute coefficients after the standardised degree-2 expansion. Interaction-term weight is shared across the original features participating in that term, so the family summary remains interpretable.

In [ ]:
fig, axes = plt.subplots(2, len(HORIZONS), figsize=(15, 7), constrained_layout=True)
for column, horizon in enumerate(HORIZONS):
    target = f"cci_t{horizon}"
    observed = validation[target]
    predicted = predictions[horizon]
    axes[0, column].scatter(observed, predicted, alpha=0.45, s=12)
    axes[0, column].plot([0, 100], [0, 100], "k--")
    axes[0, column].set(title=f"D+{horizon}: observed vs predicted", xlabel="Observed CCI", ylabel="Predicted CCI")
    axes[1, column].plot(validation.index, observed, label="observed", linewidth=1.4)
    axes[1, column].plot(validation.index, predicted, label="Polynomial Ridge prediction", linewidth=1.1)
    axes[1, column].set(title=f"{target} over validation", xlabel="Feature date", ylabel="CCI")
    axes[1, column].legend(fontsize=8)
fig.suptitle("Polynomial Ridge: validation predictions by horizon", fontsize=12)
fig.savefig(FIGURE_DIR / f"{EXPERIMENT}-validation.png", dpi=150, bbox_inches="tight")
plt.show()

polynomial = models[1].named_steps["polynomial"]
term_membership = polynomial.powers_ > 0
term_shares = term_membership / term_membership.sum(axis=1, keepdims=True)
coefficient_attribution = {}
for horizon in HORIZONS:
    coefficients = np.abs(models[horizon].named_steps["ridge"].coef_)
    coefficient_attribution[f"cci_t{horizon}"] = term_shares.T @ coefficients

importances = pd.DataFrame(coefficient_attribution, index=FEATURE_COLUMNS)
importances["mean"] = importances.mean(axis=1)
feature_family = {column: group for group, columns in FEATURE_GROUPS.items() for column in columns}
importances["family"] = [feature_family[column] for column in importances.index]
top = importances.sort_values("mean", ascending=False).head(20)
family_totals = importances.groupby("family")[[f"cci_t{h}" for h in HORIZONS]].sum()

fig, axes = plt.subplots(1, 2, figsize=(15, 6), constrained_layout=True)
top["mean"].sort_values().plot.barh(ax=axes[0])
axes[0].set(title="Top 20 features by shared absolute coefficient", xlabel="Attribution weight")
family_totals.plot.bar(ax=axes[1])
axes[1].set(title="Coefficient attribution by family and horizon", ylabel="Shared absolute coefficient", xlabel="")
fig.savefig(FIGURE_DIR / f"{EXPERIMENT}-importance.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Generalisation check and artefact export

Train and validation MAE are compared to expose any optimism gap. The final export preserves the feature order, selected regularisation, evaluation metrics, and coefficient attribution.

In [ ]:
generalisation_rows = []
for horizon in HORIZONS:
    target = f"cci_t{horizon}"
    generalisation_rows.extend([
        {"target": target, "split": "train", "mae": mean_absolute_error(train[target], models[horizon].predict(X_train))},
        {"target": target, "split": "validation", "mae": mean_absolute_error(validation[target], predictions[horizon])},
    ])
generalisation = pd.DataFrame(generalisation_rows)
print(generalisation.pivot(index="target", columns="split", values="mae"))

ax = generalisation.pivot(index="target", columns="split", values="mae").plot.bar(figsize=(8, 4.5))
ax.set(title="Polynomial Ridge train versus validation MAE", ylabel="MAE (CCI)", xlabel="Forecast target")
ax.figure.tight_layout()
ax.figure.savefig(FIGURE_DIR / f"{EXPERIMENT}-generalisation.png", dpi=150, bbox_inches="tight")
plt.show()

sys.path.insert(0, str(PROJECT_ROOT / "36120_26sp_at2_25945320_experiments"))
from modeling.deployment import export_tabular_bundle

experiment_metadata = {
    "experiment": EXPERIMENT,
    "student_id": "25945320",
    "algorithm": "Degree-2 Polynomial Ridge Regression",
    "prediction_type": "regression",
    "feature_columns": FEATURE_COLUMNS,
    "feature_column_order_is_significant": True,
    "feature_groups": FEATURE_GROUPS,
    "polynomial_degree": 2,
    "expanded_feature_count": int(polynomial.powers_.shape[0]),
    "selected_alphas": selected_alphas,
    "validation_metrics": {f"cci_t{h}": results.loc[(h, "Polynomial Ridge")].to_dict() for h in HORIZONS},
    "baseline_metrics": {f"cci_t{h}": {baseline: results.loc[(h, baseline)].to_dict() for baseline in ("Persistence", "Climatology")} for h in HORIZONS},
    "feature_importance": {target: importances[target].sort_values(ascending=False).head(20).to_dict() for target in [f"cci_t{h}" for h in HORIZONS]},
    "importance_method": "absolute standardised degree-2 Ridge coefficients; interaction weight shared among participating original features",
    "generalisation_mae": generalisation.pivot(index="target", columns="split", values="mae").to_dict(),
    "split_boundaries": metadata["split_boundaries"],
    "embargo_days": metadata["embargo_days"],
    "evaluated_on": ["train", "val"],
    "held_out": ["test"],
}
bundle_dir = export_tabular_bundle(
    output_dir=MODEL_DIR / EXPERIMENT,
    experiment=EXPERIMENT,
    models=models,
    feature_columns=FEATURE_COLUMNS,
    metadata=experiment_metadata,
    model_family="polynomial_ridge",
)
print(f"Saved API-compatible bundle to {bundle_dir.relative_to(PROJECT_ROOT)}")